In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [3]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user is inquiring about various aspects of Lunapolis, a fictional location on the moon, aiming to gather information on its capital, weather, population of cheese miners, and potential labor disputes.\n\n## SUMMARY\n- The capital of the moon is Lunapolis.\n- The weather in Lunapolis is clear, with temperatures ranging from a high of 120C to a low of -100C.\n- There are 100,000 cheese miners living in Lunapolis.\n- There is a likelihood that the cheese miners' union will strike due to dissatisfaction with the new president.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nNone", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='e9fea49a-36f0-4f75-b6ea-606631bd3ee8'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", additional_kwargs={}, response_metadata={}, id='80ce6611-f37c-4889-85d8-8

In [4]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
The user is inquiring about various aspects of Lunapolis, a fictional location on the moon, aiming to gather information on its capital, weather, population of cheese miners, and potential labor disputes.

## SUMMARY
- The capital of the moon is Lunapolis.
- The weather in Lunapolis is clear, with temperatures ranging from a high of 120C to a low of -100C.
- There are 100,000 cheese miners living in Lunapolis.
- There is a likelihood that the cheese miners' union will strike due to dissatisfaction with the new president.

## ARTIFACTS
None

## NEXT STEPS
None


## Trim/delete messages

In [5]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [7]:
agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [8]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='d5125c90-d92a-4891-b686-65bfdfcecab4'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='64b40643-662f-4507-8fac-98cf196dbdd8', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='7cc3316f-181b-4c21-9274-c6a886bbd2a4'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='bde29d3c-9b2f-473f-9654-dddb48bad08b', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='d80ba07b-e278-40c1-886c-d2e206d9865e'),
              AIMessage(content='I can’t read the temperature from here. The exact temperature 

In [8]:
print(response["messages"][-1].content)

I can’t read the internal temperature of your device from here. You’ll need to check it with the device’s sensors or a monitoring app. If you tell me what kind of device and its OS (e.g., Windows laptop, MacBook, Android phone, etc.), I can give exact steps. In the meantime, here are general tips:

- If the device feels hot to the touch:
  - Power it off and unplug it.
  - Let it cool in a well-ventilated area for 10–15 minutes.
  - Check and clear any dust from vents and fans.

- How to check temperatures (general approaches):
  - Windows PC: use HWInfo, HWiNFO64, or Core Temp to read CPU/GPU temps.
  - Mac: third-party tools like iStat Menus or Macs Fan Control.
  - Linux: run lm-sensors (sensors command) or a GUI tool like Psensor.
  - Mobile devices: apps like CPU-Z or AIDA64 can show CPU temps (availability varies by device).

- Safe temperature ranges (rough guide):
  - CPU/GPU idle: roughly 30–50°C.
  - Under load: 60–85°C is common; consistently above 90°C can be risky.
  - Bat